# SEC Filing Intelligence

*Financial Document Analysis · Information Retrieval · Grounded Answers*

---

Financial filings contain dense disclosures across companies and reporting periods. This study examines how passage retrieval, document filters, and verifiable quotations can support evidence-grounded analysis.

**Research objective:** compare retrieval strategies and analyze how source identity, passage relevance, and abstention affect answer reliability.

**Dataset status:** the bundled sample documents describe fictional companies and are used for a methods demonstration. Results obtained from these samples must not be represented as performance on real SEC filings.

### Study roadmap

1. Research question and data
2. Exploratory data analysis and quality checks
3. Feature relationships and modeling
4. Model evaluation and visual diagnostics
5. Interpretation, limitations, and conclusions

---

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "README.md").exists() and (ROOT.parent / "README.md").exists():
    ROOT = ROOT.parent
if not (ROOT / "README.md").exists():
    raise RuntimeError("Run the notebook from the project root or notebooks/ directory")
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

## 1. Read the retrieval pipeline
The corpus is validated, split into paragraphs, indexed, retrieved, then used for *extractive* citation-checked answers.

In [ ]:
from sec_filing_intelligence.corpus import load_corpus
import inspect
print(inspect.signature(load_corpus))
print(load_corpus.__doc__ or "See src/sec_filing_intelligence/corpus.py")

## 2. Inspect sample fixture data — demonstration ONLY
The records below are fictional and must not be described as financial evidence.

In [ ]:
import json
fixture = ROOT / "data" / "fixtures" / "filings.json"
rows = json.loads(fixture.read_text(encoding="utf-8"))
print("Demo records:",len(rows))
for row in rows[:3]:
    print("Fields:",sorted(row.keys()))
    print("Document ID:",row.get("document_id",row.get("id","unknown")))
    print("Source:",row.get("source_url",row.get("url","unknown")))
    print()

## 3. Run retrieval and evaluation through the implementation
No separate notebook-specific retrieval algorithm is added: the Python source is the single source of truth. These commands operate on **synthetic fixtures** and do not establish accuracy on real filings.

In [ ]:
import subprocess
env = __import__("os").environ.copy()
env["PYTHONPATH"] = str(ROOT / "src")
cmd = [sys.executable, "-m", "sec_filing_intelligence.evaluate", "--k", "3"]
result = subprocess.run(cmd,cwd=ROOT,env=env,text=True,capture_output=True)
print("Exit code:",result.returncode)
print(result.stdout[:5000])
if result.returncode: print(result.stderr[:2000])

## 4. Explore retrieval scores and failure modes
Inspect the evaluation output and the adversarial fixture list. Document identity and abstention matter at least as much as rank. This is a teaching demonstration; real-data scores remain unverified.

In [ ]:
adversarial = json.loads((ROOT/"data"/"fixtures"/"adversarial.json").read_text(encoding="utf-8"))
print("Adversarial fixture type:",type(adversarial).__name__,"count:",len(adversarial))
print("Review sources: src/sec_filing_intelligence/retrieval.py, answer.py, evaluate.py")

## 5. Transition to reusable software
- `corpus.py`: validation, normalization, paragraph boundaries.
- `retrieval.py`: overlap and BM25/bigram rank fusion.
- `answer.py`: extractive answers, citation verification, abstention.
- `evaluate.py`: repeatable retrieval evaluation.
- `acquire.py`: opt-in SEC acquisition with provenance controls.
- `tests/`: contracts and adversarial tests.

**Next real-data milestone:** independently obtain authentic 10-K/10-Q filings, confirm rights and access requirements, and build document-grounded query judgments before reporting real-data benchmark numbers.

## Extended overview — research questions

1. How do we identify the correct company, year and filing? 2. Can a simple lexical ranker retrieve relevant evidence? 3. How do we ground answers with exact text offsets? 4. When should the system abstain? 5. How does it resist instruction-like content inside retrieved text?

**Critical limitation:** The bundled teaching corpus is fictional. The visualizations below illustrate its mechanics, **not genuine financial-research performance**.

In [ ]:
# Explore fixture schema without assuming live EDGAR origin
from collections import Counter
import matplotlib.pyplot as plt
print("Fixture keys:",sorted(set().union(*(set(x) for x in rows))))
for r in rows[:3]:
    print({k:str(v)[:140] for k,v in r.items() if k not in ("text","content")})
fig,ax=plt.subplots(figsize=(7,3))
lengths=[len(str(r.get("text",r.get("content","")))) for r in rows]
ax.bar(range(1,len(lengths)+1),lengths)
ax.set(xlabel="Fictional document index",ylabel="Characters",title="Teaching fixture sizes")
plt.tight_layout();plt.show()

## Data preparation and chunking

Unlike a numerical classifier, retrieval works over text passages. Text normalization, paragraph boundaries and document identifiers are central design choices. Incorrect document identity can make a fluent answer cite the wrong filing.

In [ ]:
# Inspect actual implementation: no duplicate algorithm is maintained in the notebook.
from pathlib import Path
from itertools import islice
for fname in ["corpus.py","retrieval.py","answer.py"]:
    p=ROOT/"src"/"sec_filing_intelligence"/fname
    print("\n"+"="*12,fname,"="*12)
    print("\n".join(p.read_text(encoding="utf-8").splitlines()[:35]))

## Retrieval model comparison

The baseline counts token overlap. The hybrid method combines BM25-style lexical ranking with bigram information through rank fusion. Ranking scores are not probabilities. Evaluate both methods on *the same* questions and corpus, and ensure query labels correspond to the correct issuer and period.

In [ ]:
# Run both evaluation methods through the production module.
import json,subprocess,os
env=os.environ.copy();env["PYTHONPATH"]=str(ROOT/"src")
proc=subprocess.run([sys.executable,"-m","sec_filing_intelligence.evaluate","--k","3"],
 cwd=ROOT,env=env,capture_output=True,text=True)
if proc.returncode:
    raise RuntimeError(proc.stderr[-3000:])
print(proc.stdout[:8500])
try:
    evaluation=json.loads(proc.stdout)
    print("Evaluation top-level keys:",list(evaluation) if isinstance(evaluation,dict) else "list")
except json.JSONDecodeError:
    print("Evaluation output is not a single JSON object; inspect raw output above.")

## Retrieval diagnostics and failure analysis

Check zero-hit queries, unsupported calculations, ambiguous issuer references, old-versus-new filing periods, and adversarial instructions in document text. Evidence excerpts should be exact spans; a matching passage alone does not guarantee that a full question was answered.

In [ ]:
# Review the adversarial test cases without promoting fiction to factual evidence.
samples=adversarial if isinstance(adversarial,list) else list(adversarial.items())
print("Adversarial examples:",len(samples))
for example in samples[:5]: print(str(example)[:450],"\n---")

## Real SEC experiment protocol (not yet completed)

1. Select specific original 10-K or 10-Q documents from the SEC website and confirm issuer/accession/period.
2. Use `acquire.py` with a genuine contact user agent and comply with SEC access restrictions.
3. Retain raw and normalized hashes, licensing notes, and document provenance.
4. Draft answerable questions and independent relevance labels *before* comparing retrieval methods.
5. Compare recall-at-k, retrieval precision, abstention, citation accuracy and failure categories; report dataset size and exact file versions.

**Do not report fictional fixture metrics as performance on real SEC filings.**

In [ ]:
# Read data-acquisition contract and instructions. This does NOT download anything.
import inspect
from sec_filing_intelligence import acquire
print("Acquisition interfaces:")
for name,obj in vars(acquire).items():
    if inspect.isfunction(obj) and obj.__module__==acquire.__name__:
        print(name,inspect.signature(obj))

## Conclusion and next steps

The study remains limited to the data actually available. Complete real-data experiments and independently verified results are required before drawing deployment or domain-specific performance conclusions.

---

**Further development:** A separate modular Python implementation is included in the repository for document processing, retrieval, citation checking, and repeatable evaluation.